# H17: Gradient Anisotropy Predicts Muon Benefit

## Theoretical Motivation

The Muon optimizer applies a Newton-Schulz orthogonalization to gradient matrices before
the update step. In spectral terms, this operation **equalizes all singular values to 1**,
replacing the gradient $G = U \Sigma V^T$ with its orthogonal projection $U V^T$.

This raises a sharp falsifiable prediction: **Muon should help most when the gradient
spectrum is most anisotropic** -- i.e., when the ratio $\sigma_1 / \sigma_{\min}$ is large.
When gradients are already nearly isotropic ($\sigma_1 \approx \sigma_{\min}$), the
Newton-Schulz step changes almost nothing, so Muon should behave like plain SGD+momentum.

## Hypothesis

> **H17:** Across architectures, the gradient anisotropy at initialization
> (measured as $\sigma_1 / \sigma_{\min}$ of the first-layer gradient) positively
> correlates with Muon's training advantage over SGD ($\text{loss}_{\text{SGD}} / \text{loss}_{\text{Muon}}$).
> Spearman $\rho > 0.5$ is required to confirm.

## Why This Is a Strong Falsification Test

- If Muon's benefit came from some mechanism *other* than spectral equalization (e.g.,
  implicit learning-rate scaling, momentum interaction, numerical accident), there is
  no reason to expect correlation with gradient anisotropy.
- We test across **five qualitatively different architectures** (deep linear, ReLU MLP,
  tanh MLP, bottleneck, wide) to avoid overfitting to one geometry.
- Both optimizers get a full LR sweep, so the comparison is at each optimizer's *best* LR.

## Methodology

| Architecture    | Layers              | Activation | Expected Anisotropy |
|-----------------|---------------------|------------|---------------------|
| deep_linear     | 4 x (32, 32)       | none       | moderate            |
| relu_mlp        | 4 x (32, 32)       | ReLU       | moderate-high       |
| tanh_mlp        | 4 x (32, 32)       | tanh       | moderate            |
| bottleneck      | 32->8->32->8->32    | ReLU       | high (rectangular)  |
| wide            | 4 x (128, 128)     | none       | lower (more params) |

**Protocol:**
1. For each architecture & seed, measure gradient anisotropy at init ($\sigma_1/\sigma_{\min}$ of first-layer gradient)
2. LR-sweep SGD and Muon independently (10 and 9 candidates respectively)
3. Train 300 steps at best LR, average final loss over 5 seeds
4. Compute Muon advantage = $\text{loss}_{\text{SGD}} / \text{loss}_{\text{Muon}}$
5. Correlate anisotropy vs. advantage via Spearman $\rho$

## Environment Setup

Import NumPy for all linear algebra and random number generation. This experiment
is implemented purely in NumPy (no PyTorch/JAX) to keep the computational graph
transparent and auditable -- every gradient is computed by explicit backpropagation.

In [ ]:
import numpy as np
import os

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath('.'))


## Experimental Configuration

Key hyperparameters governing the experiment:

- **NUM_STEPS = 300**: Enough for these small networks to converge (or diverge), but short
  enough that the LR sweep over 5 architectures x 19 LR candidates x 3 seeds remains tractable.
- **MOMENTUM = 0.9**: Standard heavy-ball momentum, applied identically in both SGD and Muon
  (the only difference is whether the gradient is Newton-Schulz-orthogonalized first).
- **NS_ITERS = 5**: Newton-Schulz iterations for the polar decomposition. Five iterations
  give spectral accuracy to ~$10^{-3}$ for well-conditioned matrices.
- **NUM_SEEDS = 5**: Five independent random seeds for init + data, giving error bars.
- **BATCH_SIZE = 64**: Fixed batch (no stochastic noise within a run) to isolate optimizer
  effects from mini-batch variance.

In [ ]:
NUM_STEPS = 300
MOMENTUM = 0.9
NS_ITERS = 5
NUM_SEEDS = 5
BATCH_SIZE = 64

print("\n--- Experimental Configuration ---")
print(f"  NUM_STEPS      = {NUM_STEPS}   (training iterations per run)")
print(f"  BATCH_SIZE     = {BATCH_SIZE}    (fixed batch, no stochastic noise)")
print(f"  MOMENTUM       = {MOMENTUM}  (heavy-ball, same for SGD and Muon)")
print(f"  NUM_SEEDS      = {NUM_SEEDS}     (independent random initializations)")
print(f"  NS_ITERS       = {NS_ITERS}     (Newton-Schulz iterations for Muon)")
print(f"\n  Total LR sweep budget: 5 archs x (10+9) LR candidates x 3 seeds = {5*19*3} runs (sweep)")
print(f"  Total full-eval runs:  5 archs x 2 optimizers x 5 seeds = {5*2*5} runs (final eval)")

### Architecture and Learning Rate Search Space

Five architectures span a range of gradient geometries: square vs. rectangular weight
matrices, linear vs. nonlinear activations, narrow vs. wide layers. Each architecture
is expected to produce a different level of gradient anisotropy at initialization, giving
us the x-axis of the correlation plot.

Learning rates are swept independently for SGD and Muon to ensure a fair comparison
at each optimizer's best operating point.

In [ ]:
ARCH_NAMES = ['deep_linear', 'relu_mlp', 'tanh_mlp', 'bottleneck', 'wide']

In [ ]:
LR_SGD_CANDIDATES = [0.2, 0.1, 0.05, 0.03, 0.02, 0.01, 0.005, 0.003, 0.001, 0.0005]
LR_MUON_CANDIDATES = [0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.001]

---
## Architecture Definitions

Each architecture returns a list of weight matrix dimensions and an activation function.
The key design choices that affect gradient anisotropy:

- **deep_linear**: Pure linear chain -- gradient spectrum reflects accumulated singular value
  products across layers (multiplicative spectrum distortion).
- **relu_mlp**: ReLU kills ~50% of neurons at init, creating effective rank deficiency
  in the Jacobian and potentially increasing gradient anisotropy.
- **tanh_mlp**: tanh saturates smoothly, compressing activations and creating a gentler
  form of anisotropy than ReLU's hard gating.
- **bottleneck**: Rectangular matrices (32->8->32) force information through a low-rank
  bottleneck, which should produce **high** gradient anisotropy because the gradient
  inherits the rectangular geometry.
- **wide**: 128x128 layers have more degrees of freedom; random matrix theory predicts
  the condition number grows as $O(\sqrt{n})$, but the anisotropy per dimension is milder.

In [ ]:
def get_arch_config(arch):
    """Return (dims_list, activation) for each architecture.
    dims_list gives the weight matrix dimensions: each entry is (rows, cols).
    """
    if arch == 'deep_linear':
        # 4 layers, 32x32 each
        dims = [(32, 32)] * 4
        return dims, 'linear'
    elif arch == 'relu_mlp':
        # 4 layers, 32x32 each, ReLU activations
        dims = [(32, 32)] * 4
        return dims, 'relu'
    elif arch == 'tanh_mlp':
        # 4 layers, 32x32 each, tanh activations
        dims = [(32, 32)] * 4
        return dims, 'tanh'
    elif arch == 'bottleneck':
        # 32->8->32->8->32
        dims = [(8, 32), (32, 8), (8, 32), (32, 8)]
        return dims, 'relu'
    elif arch == 'wide':
        # 4 layers, 128x128 each
        dims = [(128, 128)] * 4
        return dims, 'linear'
    else:
        raise ValueError(f"Unknown architecture: {arch}")

In [ ]:
def init_weights(arch, seed):
    rng = np.random.RandomState(seed)
    dims, _ = get_arch_config(arch)
    weights = []
    for (rows, cols) in dims:
        # Xavier-like initialization
        scale = np.sqrt(2.0 / (rows + cols))
        W = rng.randn(rows, cols) * scale
        # Add identity-like component for square matrices
        if rows == cols:
            W += np.eye(rows) * 0.5
        weights.append(W)
    return weights

**Initialization note:** Weights use Xavier-like scaling $\sim \mathcal{N}(0, \sqrt{2/(n_{in}+n_{out})})$
plus a 0.5-identity shift for square matrices. The identity component ensures the initial
forward pass is not purely random -- it biases toward identity-like maps, which is
common in residual/deep linear network practice and prevents vanishing signal at init.

In [ ]:
def apply_activation(x, act, is_last_layer):
    if is_last_layer or act == 'linear':
        return x
    elif act == 'relu':
        return np.maximum(0, x)
    elif act == 'tanh':
        return np.tanh(x)

In [ ]:
def act_deriv(pre, act, is_last_layer):
    if is_last_layer or act == 'linear':
        return np.ones_like(pre)
    elif act == 'relu':
        return (pre > 0).astype(float)
    elif act == 'tanh':
        return 1.0 - np.tanh(pre)**2

## Forward Pass

The network computes $\hat{y} = f(x) = \phi(W_L \cdot \phi(W_{L-1} \cdots \phi(W_1 x)))$
where $\phi$ is the activation function (identity for linear, ReLU, or tanh). The last
layer always uses identity activation (no nonlinearity on the output).

In [ ]:
def forward(weights, X, arch):
    _, act = get_arch_config(arch)
    L = len(weights)
    out = X.copy()
    for idx, W in enumerate(weights):
        out = W @ out
        out = apply_activation(out, act, idx == L - 1)
    return out

## Loss Function

Mean squared error: $\mathcal{L} = \frac{1}{2N} \sum_{n=1}^{N} \|\hat{y}_n - y_n\|^2$.
The $\frac{1}{2}$ factor simplifies the gradient (cancels with the derivative of the square).

In [ ]:
def compute_loss(weights, X, Y, arch):
    pred = forward(weights, X, arch)
    return 0.5 * np.mean(np.sum((pred - Y)**2, axis=0))

## Gradient Computation (Explicit Backpropagation)

Gradients are computed by manual forward-backward passes through the network.
For each layer $l$, the gradient $\nabla_{W_l} \mathcal{L}$ is the outer product of the
backward-flowing error signal $\delta$ with the post-activation at layer $l{-}1$:

$$\nabla_{W_l} \mathcal{L} = \delta_l \cdot a_{l-1}^T$$

where $\delta$ is propagated backwards through both the weight transpose and the
activation derivative. This is standard backpropagation, implemented without
autograd to keep the computation fully transparent.

In [ ]:
def compute_gradients(weights, X, Y, arch):
    _, act = get_arch_config(arch)
    L = len(weights)
    N = X.shape[1]

    # Forward pass storing pre and post activations
    post_acts = [X.copy()]
    pre_acts = []
    out = X.copy()
    for idx, W in enumerate(weights):
        pre = W @ out
        pre_acts.append(pre)
        out = apply_activation(pre, act, idx == L - 1)
        post_acts.append(out)

    # Backward pass
    delta = (post_acts[-1] - Y) / N
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        grads[l] = delta @ post_acts[l].T
        if l > 0:
            delta = weights[l].T @ delta
            delta = delta * act_deriv(pre_acts[l-1], act, l-1 == L-1)
    return grads

## Newton-Schulz Orthogonalization (Muon's Core Operation)

The Newton-Schulz iteration computes the **polar factor** of a matrix $M = U \Sigma V^T$,
converging to $UV^T$ (all singular values set to 1). The iteration is:

$$X_{k+1} = \frac{3}{2} X_k - \frac{1}{2} X_k (X_k^T X_k)$$

starting from $X_0 = M / \|M\|_F$. This converges cubically for matrices with singular
values in $(0, \sqrt{3})$, which the Frobenius normalization ensures.

**This is the ONLY difference between SGD and Muon in this experiment**: SGD uses the raw
gradient $G$ for the momentum update, while Muon uses $\text{NS}(G)$.

In [ ]:
def newton_schulz(M, n_iters=NS_ITERS):
    norm = np.linalg.norm(M, ord='fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

---
## Training Loop and Data Generation

The training loop implements SGD+momentum and Muon+momentum with identical structure.
The only branching point is whether `newton_schulz(grads[i])` is applied before
accumulating into the momentum buffer.

**Divergence handling:** If loss exceeds $10^{10}$ or becomes non-finite, the run returns
$\infty$ and is excluded from averaging. This prevents a single divergent LR from
corrupting the sweep.

**Data generation:** Random Gaussian inputs and targets, $X, Y \sim \mathcal{N}(0, 0.3)$.
The task is pure regression on random data -- architecture geometry is the only
variable that changes across conditions.

In [ ]:
def train(weights_init, X, Y, lr, optimizer, arch):
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros_like(W) for W in weights]
    for step in range(NUM_STEPS):
        loss = compute_loss(weights, X, Y, arch)
        if not np.isfinite(loss) or loss > 1e10:
            return float('inf')
        grads = compute_gradients(weights, X, Y, arch)
        for i in range(len(weights)):
            if optimizer == 'muon':
                mom[i] = MOMENTUM * mom[i] + newton_schulz(grads[i])
            else:
                mom[i] = MOMENTUM * mom[i] + grads[i]
            weights[i] = weights[i] - lr * mom[i]
    return compute_loss(weights, X, Y, arch)

In [ ]:
def make_data(arch, seed):
    rng = np.random.RandomState(seed)
    dims, _ = get_arch_config(arch)
    input_dim = dims[0][1]   # cols of first weight = input dim
    output_dim = dims[-1][0]  # rows of last weight = output dim
    X = rng.randn(input_dim, BATCH_SIZE) * 0.3
    Y = rng.randn(output_dim, BATCH_SIZE) * 0.3
    return X, Y

---
## Anisotropy Measurement

The key independent variable: **gradient anisotropy** at initialization.

For each architecture and seed, we compute $\nabla_{W_1}\mathcal{L}$ at initialization
(before any training), take its SVD, and report the condition number
$\kappa = \sigma_1 / \sigma_{\min}$.

We use only the **first layer's gradient** because:
1. It is closest to the input and reflects the raw data geometry
2. It is the gradient that Muon would orthogonalize in the first update step
3. Using all layers would conflate the measurement with network depth effects

Values near 1.0 mean the gradient is nearly isotropic (all directions equally important).
Large values mean some gradient directions dominate, creating exactly the kind of
spectral imbalance that Newton-Schulz orthogonalization should correct.

In [ ]:
def measure_anisotropy(arch, seeds):
    """
    Measure gradient anisotropy at init: sigma_1/sigma_min of first gradient.
    Average across seeds.
    """
    anisotropies = []
    for s in seeds:
        X, Y = make_data(arch, s)
        w = init_weights(arch, s + 5000)
        grads = compute_gradients(w, X, Y, arch)
        # Use first gradient (layer 0) for anisotropy
        G = grads[0]
        sv = np.linalg.svd(G, compute_uv=False)
        if sv[-1] > 1e-15:
            aniso = sv[0] / sv[-1]
        else:
            aniso = sv[0] / 1e-15
        anisotropies.append(aniso)
    return np.mean(anisotropies), np.std(anisotropies)

---
## Spearman Rank Correlation (Dependency-Free Implementation)

Since this experiment avoids scipy, we implement Spearman's $\rho$ from scratch.
Spearman correlation is the Pearson correlation of the **rank-transformed** data,
making it robust to nonlinear monotonic relationships and outliers.

This is the right metric because:
- We care about **monotonic** relationship (more anisotropy => more benefit), not linearity
- With only 5 data points, Pearson can be dominated by a single extreme architecture
- Spearman $\rho > 0.5$ with 5 points implies the rank ordering is substantially preserved

In [ ]:
def spearman_correlation(x, y):
    """Compute Spearman rank correlation coefficient."""
    n = len(x)
    if n < 3:
        return float('nan')

    def rank_data(arr):
        order = np.argsort(arr)
        ranks = np.empty(len(arr), dtype=float)
        ranks[order] = np.arange(1, len(arr) + 1, dtype=float)
        # Handle ties by averaging ranks
        for i in range(len(arr)):
            tied = np.where(arr == arr[i])[0]
            if len(tied) > 1:
                avg_rank = np.mean(ranks[tied])
                ranks[tied] = avg_rank
        return ranks

    rx = rank_data(np.array(x, dtype=float))
    ry = rank_data(np.array(y, dtype=float))
    # Pearson on ranks
    return np.corrcoef(rx, ry)[0, 1]

---
## Main Experiment Execution

The main function orchestrates the full pipeline for all five architectures:

1. **Anisotropy measurement** -- SVD of first-layer gradient at init, averaged over seeds
2. **LR sweep** -- Independent grid search for SGD and Muon, using 3 seeds for speed
3. **Full evaluation** -- Train at best LR with all 5 seeds, compute mean final loss
4. **Advantage computation** -- Ratio $\text{loss}_{\text{SGD}} / \text{loss}_{\text{Muon}}$
5. **Correlation analysis** -- Spearman and Pearson correlations (linear and log-log)
6. **Hypothesis tests** -- Three tests for progressively finer predictions

The three hypothesis tests are:
- **T1**: Spearman $\rho > 0.5$ (overall monotonic relationship)
- **T2**: The architecture with highest anisotropy also has highest Muon advantage (rank-1 match)
- **T3**: The architecture with lowest anisotropy has below-median Muon advantage (bottom consistency)

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]

    print("=" * 100)
    print("H17: GRADIENT ANISOTROPY PREDICTS MUON BENEFIT")
    print("=" * 100)
    print(f"Architectures: {ARCH_NAMES}")
    print(f"Steps: {NUM_STEPS}, Seeds: {NUM_SEEDS}")
    print()

    arch_results = {}

    for arch in ARCH_NAMES:
        dims, act = get_arch_config(arch)
        print(f"\n{'=' * 80}")
        print(f"  ARCHITECTURE: {arch}")
        print(f"  Layers: {dims}, Activation: {act}")
        print(f"{'=' * 80}")

        # Step 1: Measure anisotropy at init
        aniso_mean, aniso_std = measure_anisotropy(arch, seeds)
        print(f"  Gradient anisotropy at init: {aniso_mean:.2f} +/- {aniso_std:.2f}")

        # Step 2: LR sweep for both optimizers
        print(f"  LR sweep...")
        best = {}
        for opt, candidates in [('sgd', LR_SGD_CANDIDATES), ('muon', LR_MUON_CANDIDATES)]:
            best_lr, best_loss = candidates[-1], float('inf')
            for lr in candidates:
                losses = []
                for s in seeds[:3]:
                    X, Y = make_data(arch, s)
                    w = init_weights(arch, s + 5000)
                    fl = train(w, X, Y, lr, opt, arch)
                    losses.append(fl)
                finite = [l for l in losses if np.isfinite(l)]
                ml = np.mean(finite) if finite else float('inf')
                if ml < best_loss:
                    best_loss = ml
                    best_lr = lr
            best[opt] = best_lr
            print(f"    {opt:>5}: best_lr={best_lr}")

        # Step 3: Full training with all seeds
        final_losses = {}
        for opt in ['sgd', 'muon']:
            losses = []
            for s in seeds:
                X, Y = make_data(arch, s)
                w = init_weights(arch, s + 5000)
                fl = train(w, X, Y, best[opt], opt, arch)
                losses.append(fl)
            finite = [l for l in losses if np.isfinite(l)]
            mean_loss = np.mean(finite) if finite else float('inf')
            std_loss = np.std(finite) if len(finite) > 1 else 0.0
            final_losses[opt] = {'mean': mean_loss, 'std': std_loss, 'lr': best[opt]}

        advantage = final_losses['sgd']['mean'] / max(final_losses['muon']['mean'], 1e-30)
        print(f"  SGD loss:  {final_losses['sgd']['mean']:.6e} (lr={final_losses['sgd']['lr']})")
        print(f"  Muon loss: {final_losses['muon']['mean']:.6e} (lr={final_losses['muon']['lr']})")
        print(f"  Muon advantage: {advantage:.2f}x")

        arch_results[arch] = {
            'anisotropy_mean': aniso_mean,
            'anisotropy_std': aniso_std,
            'sgd_loss': final_losses['sgd']['mean'],
            'muon_loss': final_losses['muon']['mean'],
            'advantage': advantage,
            'sgd_lr': final_losses['sgd']['lr'],
            'muon_lr': final_losses['muon']['lr'],
        }

    # ==========================================================================
    # CORRELATION ANALYSIS
    # ==========================================================================

    print(f"\n\n{'=' * 100}")
    print("RESULTS: Gradient Anisotropy vs Muon Advantage")
    print(f"{'=' * 100}")

    print(f"\n  {'Architecture':>15}  {'Anisotropy':>12}  {'SGD Loss':>14}  {'Muon Loss':>14}  {'Advantage':>12}")
    print("  " + "-" * 72)

    anisos = []
    advs = []
    for arch in ARCH_NAMES:
        r = arch_results[arch]
        anisos.append(r['anisotropy_mean'])
        advs.append(r['advantage'])
        print(f"  {arch:>15}  {r['anisotropy_mean']:>12.2f}  {r['sgd_loss']:>14.6e}  "
              f"{r['muon_loss']:>14.6e}  {r['advantage']:>12.2f}x")

    # Pearson correlation
    pearson_r = np.corrcoef(anisos, advs)[0, 1]
    # Spearman correlation
    spearman_r = spearman_correlation(anisos, advs)

    print(f"\n  Pearson correlation:  r = {pearson_r:.3f}")
    print(f"  Spearman correlation: r = {spearman_r:.3f}")

    # Log-scale correlations (anisotropy can span orders of magnitude)
    log_anisos = np.log10(np.array(anisos))
    log_advs = np.log10(np.clip(np.array(advs), 1e-10, None))
    pearson_log = np.corrcoef(log_anisos, log_advs)[0, 1]
    spearman_log = spearman_correlation(log_anisos, log_advs)
    print(f"  Pearson (log-log):    r = {pearson_log:.3f}")
    print(f"  Spearman (log-log):   r = {spearman_log:.3f}")

    # ==========================================================================
    # HYPOTHESIS TESTS
    # ==========================================================================

    print(f"\n\n{'=' * 100}")
    print("HYPOTHESIS TESTS")
    print(f"{'=' * 100}")

    t1 = spearman_r > 0.5
    print(f"\n  T1: Spearman correlation r > 0.5?")
    print(f"      Spearman r = {spearman_r:.3f}")
    if t1:
        print(f"      --> PASS: Higher gradient anisotropy DOES predict greater Muon benefit.")
    else:
        print(f"      --> FAIL: Anisotropy is NOT a reliable predictor of Muon benefit.")

    # T2: Rank ordering matches
    aniso_ranking = np.argsort(anisos)[::-1]  # highest aniso first
    adv_ranking = np.argsort(advs)[::-1]      # highest advantage first
    top_aniso_arch = ARCH_NAMES[aniso_ranking[0]]
    top_adv_arch = ARCH_NAMES[adv_ranking[0]]
    t2 = top_aniso_arch == top_adv_arch
    print(f"\n  T2: Highest anisotropy = highest Muon advantage?")
    print(f"      Highest anisotropy: {top_aniso_arch} ({anisos[aniso_ranking[0]]:.2f})")
    print(f"      Highest advantage:  {top_adv_arch} ({advs[adv_ranking[0]]:.2f}x)")
    print(f"      Ranking by anisotropy: {[ARCH_NAMES[i] for i in aniso_ranking]}")
    print(f"      Ranking by advantage:  {[ARCH_NAMES[i] for i in adv_ranking]}")
    print(f"      --> {'PASS' if t2 else 'FAIL'}")

    # T3: Low anisotropy => low advantage
    low_aniso_idx = aniso_ranking[-1]
    low_aniso_adv = advs[low_aniso_idx]
    median_adv = np.median(advs)
    t3 = low_aniso_adv < median_adv
    print(f"\n  T3: Lowest anisotropy has below-median Muon advantage?")
    print(f"      Lowest anisotropy: {ARCH_NAMES[low_aniso_idx]} (aniso={anisos[low_aniso_idx]:.2f}, adv={low_aniso_adv:.2f}x)")
    print(f"      Median advantage: {median_adv:.2f}x")
    print(f"      --> {'PASS' if t3 else 'FAIL'}")

    # ==========================================================================
    # CONCLUSION
    # ==========================================================================

    print(f"\n\n{'=' * 100}")
    print("CONCLUSION")
    print(f"{'=' * 100}")

    tests_passed = sum([t1, t2, t3])
    print(f"\n  Tests passed: {tests_passed}/3")
    print(f"  Spearman r = {spearman_r:.3f}")

    if t1:
        print(f"\n  CONFIRMED: Gradient anisotropy at initialization positively correlates")
        print(f"  with Muon's advantage (Spearman r = {spearman_r:.3f} > 0.5).")
        print(f"  Architectures with more anisotropic gradients benefit more from Muon's")
        print(f"  SV equalization, because there are more imbalanced SVs to correct.")
    else:
        print(f"\n  NOT CONFIRMED: Gradient anisotropy alone does not reliably predict")
        print(f"  Muon's advantage (Spearman r = {spearman_r:.3f} <= 0.5).")
        print(f"  Other factors (e.g., activation landscape, gradient alignment) may")
        print(f"  matter more than raw spectral anisotropy.")

    print(f"\n{'=' * 100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'=' * 100}")

    # ==========================================================================
    # PLOT
    # ==========================================================================
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle('H17: Gradient Anisotropy Predicts Muon Benefit\n'
                     f'{NUM_STEPS} steps, {NUM_SEEDS} seeds per architecture',
                     fontsize=13, fontweight='bold')

        colors_map = {
            'deep_linear': '#4477AA',
            'relu_mlp': '#CC3311',
            'tanh_mlp': '#228B22',
            'bottleneck': '#9933CC',
            'wide': '#FF8800',
        }

        # (a) Anisotropy vs Advantage (linear scale)
        ax = axes[0]
        for i, arch in enumerate(ARCH_NAMES):
            ax.scatter(anisos[i], advs[i], c=colors_map[arch], s=120,
                       edgecolors='black', zorder=3, label=arch)
            ax.annotate(arch, (anisos[i], advs[i]), textcoords="offset points",
                        xytext=(5, 5), fontsize=8)
        ax.set_xlabel('Gradient Anisotropy (sigma_1/sigma_min)')
        ax.set_ylabel('Muon Advantage (loss_SGD / loss_Muon)')
        ax.set_title(f'Linear Scale (Spearman r={spearman_r:.3f})')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        # Fit line
        if len(anisos) > 1:
            z = np.polyfit(anisos, advs, 1)
            p = np.poly1d(z)
            x_fit = np.linspace(min(anisos), max(anisos), 50)
            ax.plot(x_fit, p(x_fit), '--', color='gray', alpha=0.5, linewidth=1)

        # (b) Log-log scale
        ax = axes[1]
        for i, arch in enumerate(ARCH_NAMES):
            ax.scatter(anisos[i], advs[i], c=colors_map[arch], s=120,
                       edgecolors='black', zorder=3, label=arch)
            ax.annotate(arch, (anisos[i], advs[i]), textcoords="offset points",
                        xytext=(5, 5), fontsize=8)
        ax.set_xlabel('Gradient Anisotropy (sigma_1/sigma_min)')
        ax.set_ylabel('Muon Advantage (loss_SGD / loss_Muon)')
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_title(f'Log-Log Scale (Spearman r={spearman_log:.3f})')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plot_path = os.path.join(SCRIPT_DIR, 'h17_anisotropy_predicts_benefit.png')
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"\nPlot saved: {plot_path}")
    except ImportError:
        print("\nWARNING: matplotlib not available, skipping plot.")

### Diagnostic: Inspect Gradient Spectra at Initialization

Before running the full experiment, let us examine the gradient singular value spectra
for each architecture at a single seed. This gives intuition for *why* the anisotropy
differs across architectures and previews the main independent variable.

In [ ]:
print("=" * 80)
print("DIAGNOSTIC: Gradient Singular Value Spectra at Initialization (seed=42)")
print("=" * 80)
diag_seed = 42
for arch in ARCH_NAMES:
    X, Y = make_data(arch, diag_seed)
    w = init_weights(arch, diag_seed + 5000)
    grads = compute_gradients(w, X, Y, arch)
    G0 = grads[0]
    sv = np.linalg.svd(G0, compute_uv=False)
    aniso = sv[0] / max(sv[-1], 1e-15)
    dims, act = get_arch_config(arch)
    print(f"\n  {arch:>15} | shape={G0.shape} | act={act}")
    print(f"                  | sigma_1={sv[0]:.6f}  sigma_min={sv[-1]:.6f}  ratio={aniso:.2f}")
    print(f"                  | all SVs: {np.array2string(sv, precision=5, separator=', ')}")
    print(f"                  | effective rank (Shannon) = {np.exp(-np.sum((sv/sv.sum()) * np.log(sv/sv.sum() + 1e-30))):.2f}")
print("\n" + "=" * 80)
print("Interpretation: Higher sigma_1/sigma_min => more anisotropic gradient =>")
print("Muon's Newton-Schulz step changes the update direction more => expect more benefit.")
print("=" * 80)

In [ ]:
if __name__ == '__main__':
    main()

### Interpretation of Results

**Reading the output above:**

- The **results table** shows each architecture's gradient anisotropy, SGD loss, Muon loss,
  and the Muon advantage ratio. An advantage > 1.0 means Muon won; > 2.0 means Muon
  achieved less than half the loss of SGD.

- **T1 (Spearman > 0.5)** is the primary test. If it passes, gradient anisotropy is a
  reliable predictor of Muon's benefit across architectures.

- **T2 and T3** are secondary: they check whether the prediction holds at the extremes
  (highest and lowest anisotropy).

- The **log-log correlations** test whether the relationship is better described as a
  power law ($\text{advantage} \propto \text{anisotropy}^\alpha$) rather than linear.

**What would falsify H17?**
A Spearman $\rho \leq 0.5$ would indicate that knowing the gradient anisotropy does NOT
tell you whether Muon will help. This would challenge the spectral equalization narrative
and suggest other mechanisms (e.g., implicit LR scaling, curvature adaptation) are dominant.

## Conclusions

### What This Experiment Tests

**H17** asks whether gradient anisotropy -- the spectral condition number of the
gradient matrix at initialization -- is a reliable **predictor** of how much Muon
outperforms SGD. This is a direct consequence of the "Muon as RG gauge fixing"
framework: if Muon's benefit comes from singular value equalization, then architectures
whose gradients have more singular values to equalize should benefit more.

### Interpretation Guide

- **Spearman $\rho > 0.5$** confirms the monotonic relationship (T1 PASS)
- **Top-1 rank match** confirms the strongest prediction (T2 PASS)
- **Bottom consistency** confirms the prediction works at both extremes (T3 PASS)
- All three passing would be strong evidence for the spectral equalization mechanism

### Caveats and Limitations

1. **Only 5 architectures** -- the correlation has limited statistical power. The
   Spearman threshold of 0.5 is generous; with 5 points, even $\rho = 0.9$ can arise
   by chance with probability ~5%.
2. **First-layer gradient only** -- anisotropy at deeper layers may differ and may
   matter more for the overall training dynamics.
3. **Random data** -- real tasks have structured data geometry that interacts with
   the gradient spectrum. Anisotropy on real data may predict benefit differently.
4. **Fixed training budget** -- 300 steps may favor one optimizer over another depending
   on convergence speed vs. final accuracy tradeoffs.

### Connection to the Broader Framework

If confirmed, H17 provides a **predictive tool**: before training, one can measure
gradient anisotropy and estimate whether switching from SGD to Muon is worthwhile.
This connects to the RG gauge-fixing interpretation where Muon removes "spectral gauge
redundancy" -- the benefit is proportional to the amount of redundancy present.